In [ ]:
# ──────────────────────────────────────────────────────────────
# Headers are in the code because they look nice
# "If my commit messages don't have emojis, how would you know how I feel?"
# - ProgrammersAreAlsoHuman, '0.1x engineers'
# ──────────────────────────────────────────────────────────────

"""
This code trains several models to perform a 3SUM task generated
by https://github.com/JacobPfau/fillerTokens/tree/master, comparing
model training curves, attention maps, and test set performance in
order to highlight the role of filler tokens.


I plan to use these training runs:
    1) CoT as filler (filler unmasked)
    2) Dots as filler (filler masked out)
    3) Low entropy tokens as filler (filler masked out)
    4) High entropy tokens as filler (filler masked out)
    5) Direct-to-answer

This allows for
    1) A high-quality gold standard for the model, direct-to-answer.  Not very related to my experiment but a super-baseline.
    2) Progressively more difficult tasks to pull out some sort of linear relationship (heavily relies on the assumption that I've picked tokens which are linear along some x axis, but these *were* calculated via the entropix repo as progressively more-difficult-to-predict tokens.  Need to formalize this more but for now not worth the time).  Keep the experiment simple as there are so many possible experiments.


Outline of data pipeline for clarity:
1) We generate three kinds of samples, direct, CoT and punct:
    533 569 530 814 A False     VERIFY THIS DOESNT HAVE P AND A IM NOT SURE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    371 578 006 519 P 1- 4 0- 7 3- 7- 4- 4 5- 6 A True
    873 545 827 245 P . . . . . . . . . . . . . A False
    # Note that this is prompt followed by answer in the same string
    # Answers are one word long (True/False), so
      future [-1], len(prompt)-1, are referring to this
2) Turn that into a HF dataset which has two columns:
    a) input_ids: tokenized full prompts
    b) labels: same except -100-masked everywhere but answer and eos tokens
3) Lastly, the collator recieves a list of those dicts and
    a) pads batches to the longest of that batch
    b) stacks examples into tensors
    c) leaves labels alone
"""


In [1]:
# ──────────────────────────────────────────────────────────────
# Installs
# ──────────────────────────────────────────────────────────────
# I don't think we need these:
# !pip uninstall -y torch torchvision torchaudio
# !pip install -U bitsandbytes accelerate transformers peft

# Try/except to avoid repeated install; not sure of a better way
try:
    import bitsandbytes as bnb
except:
    # You can tell I'm having a bad time:
    # !pip install --prefer-binary bitsandbytes
    # !pip uninstall -y flash-attn -q
    # !pip install --no-binary :all: --no-build-isolation flash-attn
    # !pip uninstall -y datasets fsspec gcsfs
    # !pip install -U "datasets>=2.19.1" "fsspec>=2024.3.1"
    # !pip install -U flash-attn --no-build-isolation
    # !pip install --prefer-binary torch==2.5.0+cu124 torchvision==0.18.0+cu124 torchaudio==2.5.0+cu124 -f https://download.pytorch.org/whl/cu124/torch_stable.html bitsandbytes
    # !pip install --prefer-binary torch==2.5.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121 bitsandbytes
    !pip uninstall -y torch torchvision torchaudio -q
    !pip install --index-url https://download.pytorch.org/whl/cu121 \
        torch==2.3.0 torchvision torchaudio
    !pip install flash-attn==2.7.4.post1 --no-build-isolation
    !pip install -U bitsandbytes==0.45.5
    !pip install -U transformers accelerate peft datasets fsspec tqdm
    !pip install -U "fsspec==2025.3.2"


In [2]:
# ──────────────────────────────────────────────────────────────
# Imports
# ──────────────────────────────────────────────────────────────
import datetime as dt
import itertools
import json
import os
import tempfile
import pathlib
import shutil
import re
import textwrap
import subprocess
import shlex
from pathlib import Path
from glob import glob
from functools import partial
from typing import Final
from types import SimpleNamespace as ns
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset, load_from_disk
from google.colab import drive
from huggingface_hub import notebook_login
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score


In [3]:
# ──────────────────────────────────────────────────────────────
# Global variables and namespace for the run
# ──────────────────────────────────────────────────────────────
# Run namespace
presets = {
    'cot': ns(
        runtype='cot',
        save_tag=None,
        filler_tok=None
    ),
    'direct': ns(
        runtype='direct',
        save_tag=None,
        filler_tok=None
    ),
    'filler_space-dot': ns(
        runtype='filler',
        save_tag='space-dot',
        filler_tok=' .'
    ),
    'filler_dot': ns(
        runtype='filler',
        save_tag='dot',
        filler_tok='.'
    ),
    'filler_space-neu': ns(
        runtype='filler',
        save_tag='space-neu',
        filler_tok=' neu'
    ),
    'filler_space-big': ns(
        runtype='filler',
        save_tag='space-big',
        filler_tok=' big'
    )
}
cfg = presets['cot']

# Core paths
BASE_DIR = "/content/drive/MyDrive/Colab_Files/repurposed_tokens"
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODEL_ID = "meta-llama/Llama-3.2-1B"  # or 3-8B
MODEL_NAME = MODEL_ID.split('/')[-1]
tag = f"/{cfg.save_tag}" if cfg.save_tag else ""
MODEL_DIR = f"{BASE_DIR}/models/{MODEL_NAME}/{cfg.runtype}{tag}"
print("Model dir: ", MODEL_DIR)

# Set data paths
tag = f"/{cfg.save_tag}" if cfg.save_tag else ""
cfg.data_dir = f"{BASE_DIR}/data/{cfg.runtype}{tag}"
print(cfg.runtype, cfg.data_dir)

cfg.train_tok = f"{cfg.data_dir}/train_tokenized"
cfg.test_tok = f"{cfg.data_dir}/test_tokenized"

# GPU setup (T4 vs A100)
gpu_presets = {
    't4': ns(
        torch_dtype=torch.float16,
        bnb_4bit_compute_dtype=torch.float16,
        attn_implementation='sdpa',
        bf16=False,  # Not possible on T4s
        fp16=True,
        fp16_full_eval=True,
        grad_ckpt=False,
        pad_to_multiple_of=None,
        tf32=False  # Not possible on T4s
    ),
    'a100': ns(
        torch_dtype=torch.bfloat16,
        bnb_4bit_compute_dtype=torch.bfloat16,
        attn_implementation='flash_attention_2',
        bf16=True,
        fp16=False,
        fp16_full_eval=False,
        grad_ckpt=False,
        pad_to_multiple_of=8,  # Uses TensorCores better on A100s
        tf32=True  # Only means stragglers use it - not converting anything
    )
}
gpu = gpu_presets['a100']


Model dir:  /content/drive/MyDrive/Colab_Files/repurposed_tokens/models/Llama-3.2-1B/cot
cot /content/drive/MyDrive/Colab_Files/repurposed_tokens/data/cot


In [4]:
# ──────────────────────────────────────────────────────────────
# Github and Drive setup
# ──────────────────────────────────────────────────────────────
%cd /content

# Mount drive for data
drive.flush_and_unmount()
if os.path.exists('/content/drive') and not os.path.islink('/content/drive'):
    shutil.rmtree('/content/drive')
drive.mount('/content/drive', force_remount=True)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(cfg.data_dir, exist_ok=True)

# Filepaths
FILLER_DIR = "/content/fillerTokens"
ENTROPIX_DIR = "/content/entropix"

# Clone and setup repos
if not os.path.exists(FILLER_DIR):
    !git clone https://github.com/BreckEmert/fillerTokens.git
    !pip install -r {FILLER_DIR}/requirements.txt

if not os.path.exists(ENTROPIX_DIR):
    !git clone https://github.com/BreckEmert/entropix.git

# Set HF cache
os.environ['HF_HOME'] = '/content/drive/MyDrive/HF_cache'


/content
Mounted at /content/drive


In [5]:
# ──────────────────────────────────────────────────────────────
# HF setup
# ──────────────────────────────────────────────────────────────
notebook_login()


# Loading the Tokenizer

In [6]:
# ──────────────────────────────────────────────────────────────
# Prepare tokenizer
# ──────────────────────────────────────────────────────────────
# Note: We only want loss on the answer token and eos, because we want it to use the periods how it wants, not generate them.
tok = AutoTokenizer.from_pretrained(
    os.path.join(BASE_DIR, 'tokenizer'),
    use_fast=True
)

# Set the custom attributes again
tok.pad_token = tok.eos_token
tok.cot_begin_id = tok(" P", add_special_tokens=False)["input_ids"][0]
tok.cot_end_id = tok(" A", add_special_tokens=False)["input_ids"][0]
tok.true_id = tok(" True", add_special_tokens=False)["input_ids"][0]
tok.false_id = tok(" False", add_special_tokens=False)["input_ids"][0]

if cfg.filler_tok:
    cfg.filler_id = tok(
        cfg.filler_tok, add_special_tokens=False
    )["input_ids"][0]

    cfg.filler_id = torch.tensor(cfg.filler_id)


# Loading the Data

In [7]:
train_ds = load_from_disk(cfg.train_tok)  #.with_format("torch")
test_ds  = load_from_disk(cfg.test_tok )  #.with_format("torch")


# Config/Collator for both Finetune AND Eval Runs

In [1]:
# ──────────────────────────────────────────────────────────────
# Config/Collator for both finetune AND eval runs
# ──────────────────────────────────────────────────────────────
# Bnb config
# 4 bit
# bnb_cfg = BitsAndBytesConfig(
#     load_in_4bit=gpu.load_in_4bit,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=gpu.bnb_4bit_compute_dtype,
# )
# 8 bit:
bnb_cfg = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,  # CHANGE IF OVERFLOW
    llm_int8_enable_fp32_cpu_offload = False
)

# Just pad/stacks
def pad_batch(batch, pad_id, m=None):
    # Regular pad-to-minimum
    batch = pad_sequence(
        [seq if isinstance(seq, torch.Tensor)
         else torch.tensor(seq) for seq in batch],
        batch_first=True,
        padding_value=pad_id
    )

    # A100 multiple of 8 efficiency
    if m:
        pad_right = (-batch.size(1)) % m
        if pad_right:
            batch = F.pad(batch, (0, pad_right), value=pad_id)
    return batch


def custom_data_collator(examples):
    input_ids = pad_batch(
        [ex['input_ids'] for ex in examples],
        tok.pad_token_id,
        m=gpu.pad_to_multiple_of
    )
    labels = pad_batch(
        [ex['labels'] for ex in examples],
        -100,
        m=gpu.pad_to_multiple_of
    )
    return {
        'input_ids': input_ids,
        'labels': labels,
        'attention_mask': (input_ids != tok.pad_token_id).long()
    }


NameError: name 'BitsAndBytesConfig' is not defined

# Finetuning

In [9]:
# ──────────────────────────────────────────────────────────────
# Load the base model
# ──────────────────────────────────────────────────────────────
# Load model and resize embeddings
# !pip install -U flash-attn --no-build-isolation
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    torch_dtype=gpu.torch_dtype,
    device_map='auto',
    attn_implementation=gpu.attn_implementation,
)
base_model.resize_token_embeddings(len(tok))
base_model.config.use_cache = False  # T4 small mem + grad_ckpt doesn't allow

# Enabling A100-specific settings
torch.backends.cuda.matmul.allow_tf32 = gpu.tf32
# if gpu.grad_ckpt:
#     base_model.gradient_checkpointing_enable(
#         gradient_checkpointing_kwargs={"use_reentrant": False}
#     )


# ──────────────────────────────────────────────────────────────
# Attach LoRA adapter to the model
# ──────────────────────────────────────────────────────────────
# LoRA config
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
)
# You may wonder if targeting attn is self-fulfilling for my experiment.  It is a decent bias, but, because attention mass != usefulness, indirect paths exist, and the rest of the model must adapt to any changes in attn, it's not locking in the results.  I should test LoRA on MLP and compare, ideally, which maybe I'll get around to but I only have so much compute and time.
# MLP would be gate_proj, up_proj, down_proj
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# Now that the model is made we can move this to the gpu
if cfg.filler_tok:
    cfg.filler_id = cfg.filler_id.to(model.device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [10]:
# ──────────────────────────────────────────────────────────────
# Finetune
# ──────────────────────────────────────────────────────────────
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    mask = shift_labels != -100
    pred_ids = shift_logits.argmax(-1)[mask]
    label_ids = shift_labels[mask]
    acc = accuracy_score(label_ids, pred_ids)
    return {"accuracy": acc}

tiny_eval = test_ds.shuffle(seed=345978).select(range(16))

args = TrainingArguments(
    output_dir=os.path.join(MODEL_DIR, 'checkpoints'),
    label_names=['labels'],
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # 8*4 = 32 examples per opt step
    learning_rate=8e-5,
    lr_scheduler_type='cosine',
    max_steps=80,
    warmup_steps=6,
    bf16=gpu.bf16,
    fp16=gpu.fp16,
    fp16_full_eval=gpu.fp16_full_eval,
    optim='paged_adamw_8bit',
    logging_steps=10,  # How often to print training loss
    eval_strategy='steps',
    eval_steps=10,  # How often to print accuracy
    load_best_model_at_end=True,
    metric_for_best_model='eval_accuracy',
    greater_is_better=True,
    save_strategy='steps',
    save_steps=20,
    report_to='none',  # DISABLE WANDB FOR QUICK cfgS
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=tiny_eval,
    data_collator=custom_data_collator,
    compute_metrics=compute_metrics
)

# Train
trainer.train()
model.save_pretrained(MODEL_DIR, safe_serialization=True)
    # Note theoretical CE loss, ln(2)=0.6931

final_metrics = trainer.evaluate(test_ds)
print(final_metrics['eval_accuracy'])


Step,Training Loss,Validation Loss,Accuracy
10,8.820200,4.897090,0.000000
20,1.901100,0.742226,0.562500
30,0.663900,0.385756,0.750000
40,0.183400,0.008941,1.000000
50,0.154100,0.003117,1.000000
60,0.039400,0.000777,1.000000
70,0.000600,0.000555,1.000000
80,0.054400,0.000526,1.000000


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.60 GiB. GPU 

# Evaluation and Visualization

In [ ]:
# ──────────────────────────────────────────────────────────────
# Load the finetuned model and fuse in the LoRA adapter
# ──────────────────────────────────────────────────────────────
# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    torch_dtype='auto',
    device_map='auto',
    attn_implementation='eager',  # for pure accuracy eval we can do sdpa on T4 or flash_attention_2 on A100
)

# Finetuned LoRA
model = PeftModel.from_pretrained(
    base_model,
    MODEL_DIR,
    is_trainable=False,
    torch_dtype='auto',
    device_map='auto',
)

# Merge
model = model.merge_and_unload()
model.eval()

# Compile
model = torch.compile(model, mode="reduce-overhead")
model.config.pad_token_id = tok.pad_token_id


In [ ]:
# ──────────────────────────────────────────────────────────────
# Attention hook
# ──────────────────────────────────────────────────────────────
def _record_attn(layer_idx, module, input, output):
    """Writes (B, H) into CURRENT_FILL (B, L, H)"""
    attention_probs = output[1]  # (B, H, Q, K)

    filler_mask = torch.isin(
        TOKEN_IDS_THIS_BATCH.to(model.device),
        cfg.filler_id
    )[:, None, None, :]  # (B, 1, 1, K)
        # B matches
        # 1 broadcasts across all heads
        # 1 broadcasts across all query positions
        # K matches

    # Compute mean over query dimension
    fill = (attention_probs * filler_mask).sum(-1).mean(-1)  # (B, H)
    CURRENT_FILL[:, layer_idx, :] = fill.cpu()

    del attention_probs
    torch.cuda.empty_cache()

# Attach to every layer
def attach_attention_hooks(model):
    return [
        block.self_attn.register_forward_hook(partial(_record_attn, layer_idx)) for layer_idx, block in enumerate(model.model.layers)
    ]

L = model.config.num_hidden_layers
H = model.config.num_attention_heads
handles = attach_attention_hooks(model)


In [ ]:
# ──────────────────────────────────────────────────────────────
# Attention logging and visualization
# ──────────────────────────────────────────────────────────────
def get_raw_attention(dataloader):
    global TOKEN_IDS_THIS_BATCH, CURRENT_FILL
    ATTN_BUCKET.clear()

    with torch.no_grad():
        for batch in tqdm(dataloader):
            TOKEN_IDS_THIS_BATCH = batch["input_ids"].to(model.device)  # (B, S)

            B = TOKEN_IDS_THIS_BATCH.size(0)  # L and H are global
            CURRENT_FILL = torch.zeros(B, L, H)  # (B, L, H)
                # Remember CURRENT_FILL is filled by the hooks we set
            _ = model(
                **{k: v.to("cuda") for k, v in batch.items()},
                use_cache=False,
                output_attentions=True
            )
            ATTN_BUCKET.append(CURRENT_FILL)

    raw_attn = torch.cat(ATTN_BUCKET, dim=0)  # (N, L, H)
    return raw_attn

def calculate_enrichment(raw_attn):
    """Adjusts the attention for the percentage of filler tokens
    in the ds otherwise the attention mass is biased
    """
    # Calculate the filler rate
    n_fill, n_total = 0, 0
    for ex in test_ds:
        ids = ex["input_ids"]
        n_total += len(ids)
        n_fill += torch.isin(torch.tensor(ids), cfg.filler_id).sum().item()

    fill_rate = n_fill / n_total if n_total else 0.0
    print(f"Corpus filler rate: {fill_rate:.3%}")
        # THIS MIGHT BE WRONG FOR DIRECT?  BCAUSE THERES NO FILLER TO DILUTE  not sure what I'm looking for with direct

    # Calculate the enrichment
    prob_to_fill = raw_attn.mean(dim=0)  # (L, H)
    return prob_to_fill / max(fill_rate, 1e-9)

def plot_attention(mat, title=''):
    """Quick heat-map"""
    import matplotlib.pyplot as plt
    plt.imshow(mat.numpy(), aspect='auto')
    plt.xlabel("Head")
    plt.ylabel("Layer (0 = bottom)")
    plt.title(title)
    plt.colorbar(label="× corpus filler rate")
    plt.show()

# Get raw attention
ATTN_BUCKET = []
raw_attn = get_raw_attention(
    torch.utils.data.DataLoader(
        test_ds.shuffle().select(range(500)),
        batch_size=8,
        shuffle=False,
        collate_fn=custom_data_collator,
    )
)

# Convert to enrichment
    # I regret writing mean_attn because I don't even think I want these plots yet for direct runs.  Might be used later, I guess.  But for run = 'direct' I'm just moving to accuracy.
if cfg.filler_id is not None:
    enr = calculate_enrichment(raw_attn)
    plot_attention(enr, f"Enrichment - {cfg.save_tag}")
    np.save(os.path.join(MODEL_DIR, "enrichment.npy"), enr.numpy())
else:
    mean_attn = raw_attn.mean(dim=0)
    plot_attention(mean_attn, f"Mean Attention - {cfg.save_tag}")
    np.save(os.path.join(MODEL_DIR, "mean_attn.npy"), mean_attn.numpy())


In [ ]:
# ──────────────────────────────────────────────────────────────
# Accuracy Eval
# ──────────────────────────────────────────────────────────────
# Remove these if they're still on
for handle in handles:
    handle.remove()

def accuracy(model, testset) -> float:
    """Evaluates model accuracy on test set"""
    # Iterate over test set
    invalid, correct = Counter(), 0
    with torch.no_grad():
        for ex in tqdm(testset, leave=False):
            # Forward the input to get an output
            ids = torch.tensor(
                ex["input_ids"], device=model.device
            ).unsqueeze(0)
            mask = torch.ones_like(ids)
            print(ids)
            print(tok.decode(ids[0], skip_special_tokens=False))

            break
            # keep only the prompt up to (and incl.) the " P" delimiter, drop the filler stretch
            cut = (ids[0] == tok.cot_begin_id).nonzero(as_tuple=True)[0] + 1
            ids, mask = ids[:, :cut], mask[:, :cut]
            print(ids)
            # break

            out = model.generate(
                input_ids=ids[:, :-1],  # Don't give it the answers
                attention_mask=mask[:, :-1],
                max_new_tokens=1,
                do_sample=False,
                temperature=None,
                top_p=None,  # Trying these to None because warnings?
            )
            print(out)
            break

            # Check answer
            pred_id = out[0, -1].item()
            exp_id = ids[0, -1].item()

            if pred_id == exp_id:
                correct += 1
            elif pred_id not in [tok.true_id, tok.false_id]:
                invalid[pred_id] += 1

    accuracy = correct / len(testset)
    return accuracy, invalid

# this is off now that I don't have puncts/no puncts since that would be handled by aggregating several runs of data, not something we do all at once necessarily?  I guess we can do it all at once after all the models are trained, but it seems useful to have some sort of utility function that works as we go so we can tune the runs.
acc_puncts, invalid_puncts = accuracy(
    model, test_ds.shuffle().select(range(100))
)

print(f"Invalid count: {sum(invalid_puncts)}")
print(f"Accuracy: {acc_puncts:.3f}")


# Debug Cells

In [ ]:
# Temp debugging cell
# AI GENERATED CELL
mc = model.config  # shorthand

# Ensure pad_token_id is set
if mc.pad_token_id is None:
    mc.pad_token_id = tok.pad_token_id

print(f"{'Name':<18}│ {'ID':<7}│ Token")
print("────────────────────┼─────────┼──────────────────────")
for name, tid in relevant_ids.items():
    tok_str = tok.convert_ids_to_tokens([tid], skip_special_tokens=False)[0]
    print(f"{name:<18}│ {tid:<7}│ {repr(tok_str)}")


In [ ]:
# AI GENERATED DEBUG CELL
from itertools import islice

N_EXAMPLES = 2          # how many rows of the HF dataset to inspect
MAX_COLS   = 120        # crop very long prompts for readability

for ex_idx in range(N_EXAMPLES):
    ids   = train_ds[ex_idx]['input_ids']
    lbls  = train_ds[ex_idx]['labels']
    toks  = tok.convert_ids_to_tokens(ids, skip_special_tokens=False)

    print(f"\n── Example {ex_idx} ──")
    print("pos | token                    | id         | label_id")
    print("----+---------------------------+------------+----------")
    for pos, (tok_str, tok_id, lab_id) in enumerate(zip(toks, ids, lbls)):
        print(f"{pos:3d} | {tok_str:<25.25} | {tok_id:<10d} | {lab_id}")
        if lab_id != -100 and lab_id != tok_id:
            raise ValueError(
                f"Label/input mismatch at pos {pos}: "
                f"input={tok_id}, label={lab_id}"
            )

print("\n✓ All inspected examples have self-consistent masks.")


In [ ]:
# AI GENERATED CELL
import torch
from itertools import islice

N_TO_SHOW = 1           # how many examples to dump

def show_eval_io(model, ds, n=N_TO_SHOW):
    """
    Prints, for n examples:
      • input token, input id
      • predicted token, predicted id
      • expected (last prompt token) token & id
    """
    for ex_num, ex in enumerate(islice(ds.shuffle(seed=0), n)):
        ids  = torch.tensor(ex["input_ids"], device=model.device).unsqueeze(0)
        mask = torch.ones_like(ids)

        # === generate one token (same args as in your accuracy()) ============
        out = model.generate(
            input_ids       = ids,
            attention_mask  = mask,
            max_new_tokens  = 1,
            do_sample       = False,
            temperature     = None,
            top_p           = None,
        )

        pred_id  = out[0, -1].item()
        exp_id   = ids[0, -1].item()

        pred_tok = tok.convert_ids_to_tokens([pred_id],  skip_special_tokens=False)[0]
        exp_tok  = tok.convert_ids_to_tokens([exp_id],   skip_special_tokens=False)[0]

        # === pretty print ====================================================
        print(f"\n━━ Example {ex_num} ━━")
        print("pos │ token                    │ id")
        print("───┼───────────────────────────┼────────")
        for pos, tok_id in enumerate(ids[0].tolist()):
            tok_str = tok.convert_ids_to_tokens([tok_id], skip_special_tokens=False)[0]
            print(f"{pos:3d} │ {tok_str:<25.25} │ {tok_id}")

        print("───┴───────────────────────────┴────────")
        print(f"Expected (last prompt token): {exp_tok!r:25} id={exp_id}")
        print(f"Predicted by model          : {pred_tok!r:25} id={pred_id}")
        verdict = "✓ MATCH" if pred_id == exp_id else "✗ MISMATCH"
        print("Result:", verdict)

# call it
show_eval_io(model, test_ds, n=N_TO_SHOW)
